# Chapter 7 — qcirclab version

The focus is on logical circuit metrics, manual optimization, toy routing, basis rewriting, and resource accounting. Unlike a production transpiler, the routines below are intentionally small and explicit. They are meant to show the ideas behind compilation rather than to target a real backend.

In [8]:
# Colab installation
# Uncomment when running in a fresh Colab runtime.
!pip install -q --no-cache-dir --force-reinstall git+https://github.com/2forts/qcirclab_repo.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 141.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


In [9]:
import numpy as np
import time
from collections import deque

from qcirclab import Circuit
import qcirclab.gates as qg
from qcirclab.operators import (
    append_operation,
    circuit_unitary,
    equal_up_to_global_phase,
    circuit_without_measurements,
)
from qcirclab.metrics import circuit_metrics, print_metrics

## Subsection 7.2.5 — Practical evaluation without a backend transpiler

In [10]:
# Example circuit: three qubits and a final measurement layer.

qc = Circuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.barrier()
qc.rx(0.4, 0)
qc.ry(0.2, 1)
qc.rz(0.1, 2)
qc.measure_all()

print(qc.draw())
print_metrics("Logical circuit", qc)

q0: |0>[ H ]─●──── ┆ [ RX]───  ───  ─M───────
q1: |0>───  ─X──●─ ┆ ───  [ RY]───  ────M────
q2: |0>───  ────X─ ┆ ───  ───  [ RZ]───────M─
c0:  0                              ═╩═══════
c1:  0                              ════╩════
c2:  0                              ═══════╩═
=== Logical circuit ===
qubits: 3
classical_bits: 3
operations: 9
depth: 6
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 1, 'cx': 2, 'rx': 1, 'ry': 1, 'rz': 1, 'measure': 3}



In [11]:
# A simple backend-independent timing estimate.
# These numbers are illustrative only.

gate_durations_ns = {
    "h": 35,
    "x": 35,
    "rx": 35,
    "ry": 35,
    "rz": 0,
    "s": 0,
    "t": 0,
    "cx": 300,
    "cz": 300,
    "swap": 900,
    "measure": 1000,
}

def estimate_duration_ns(qc, durations):
    qtime = [0.0] * qc.n_qubits
    ctime = [0.0] * qc.n_clbits

    for op in qc.operations:
        if op.name == "barrier":
            m = max(qtime + ctime) if (qtime or ctime) else 0
            qtime = [m] * qc.n_qubits
            ctime = [m] * qc.n_clbits
            continue

        used_q = set(op.targets) | set(op.controls)
        used_c = set(op.ctargets)
        if op.condition is not None:
            used_c.add(op.condition.bit)

        start = 0.0
        if used_q:
            start = max(start, max(qtime[q] for q in used_q))
        if used_c:
            start = max(start, max(ctime[c] for c in used_c))

        duration = durations.get(op.name, 50)
        finish = start + duration

        for q in used_q:
            qtime[q] = finish
        for c in used_c:
            ctime[c] = finish

    return max(qtime + ctime) if (qtime or ctime) else 0.0

print("Estimated logical duration (ns):", estimate_duration_ns(qc, gate_durations_ns))

Estimated logical duration (ns): 1670.0


## Subsection 7.3.1 — Logical vs. physical circuits

In [12]:
# Logical circuit with a long-range interaction.

logical = Circuit(3, 3)
logical.h(0)
logical.cx(0, 2)
logical.cx(1, 2)
logical.barrier()
logical.measure_all()

print("Logical circuit:")
print(logical.draw())
print_metrics("Logical", logical)

Logical circuit:
q0: |0>[ H ]─●──── ┆ ─M───────
q1: |0>───  ────●─ ┆ ────M────
q2: |0>───  ─X──X─ ┆ ───────M─
c0:  0               ═╩═══════
c1:  0               ════╩════
c2:  0               ═══════╩═
=== Logical ===
qubits: 3
classical_bits: 3
operations: 6
depth: 5
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 1, 'cx': 2, 'measure': 3}



In [13]:
def shortest_path(num_qubits, edges, start, goal):
    adj = {q: [] for q in range(num_qubits)}
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)

    queue = deque([(start, [start])])
    seen = {start}

    while queue:
        node, path = queue.popleft()
        if node == goal:
            return path
        for nxt in adj[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, path + [nxt]))

    raise ValueError(f"No path between qubits {start} and {goal}")


def has_edge(edges, a, b):
    return (a, b) in edges or (b, a) in edges


def route_cx_to_coupling(qc: Circuit, edges) -> Circuit:
    # Toy router: replace non-adjacent CX gates by SWAP chains on an undirected topology.
    routed = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_routed")

    for op in qc.operations:
        if op.name == "cx" and len(op.controls) == 1 and len(op.targets) == 1:
            c = op.controls[0]
            t = op.targets[0]

            if has_edge(edges, c, t):
                routed.cx(c, t)
            else:
                path = shortest_path(qc.n_qubits, edges, c, t)

                # Move the control state next to the target, apply CX, then restore.
                for i in range(len(path) - 2):
                    routed.swap(path[i], path[i + 1])

                routed.cx(path[-2], path[-1])

                for i in reversed(range(len(path) - 2)):
                    routed.swap(path[i], path[i + 1])
        else:
            append_operation(routed, op)

    return routed


line_edges_3 = [(0, 1), (1, 2)]

physical = route_cx_to_coupling(logical, line_edges_3)

print("Physical/routed circuit on a line topology:")
print(physical.draw())

print_metrics("Logical", logical)
print_metrics("Routed", physical)

Physical/routed circuit on a line topology:
q0: |0>[ H ]─x─────x──── ┆ ─M───────
q1: |0>───  ─x──●──x──●─ ┆ ────M────
q2: |0>───  ────X─────X─ ┆ ───────M─
c0:  0                     ═╩═══════
c1:  0                     ════╩════
c2:  0                     ═══════╩═
=== Logical ===
qubits: 3
classical_bits: 3
operations: 6
depth: 5
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 1, 'cx': 2, 'measure': 3}

=== Routed ===
qubits: 3
classical_bits: 3
operations: 8
depth: 7
two_qubit_gates: 4
multi_qubit_gates: 0
gate_counts: {'h': 1, 'swap': 2, 'cx': 2, 'measure': 3}



## Subsection 7.3.2 — A simple pass pipeline

In [14]:
# Circuit with redundant and simplifiable operations.

qc_pass = Circuit(2, 2)
qc_pass.h(0)
qc_pass.h(0)          # cancels
qc_pass.rx(0.3, 0)
qc_pass.rx(-0.3, 0)   # cancels
qc_pass.rz(0.2, 0)
qc_pass.rz(0.4, 0)    # fuses with previous Rz
qc_pass.cx(0, 1)
qc_pass.cx(0, 1)      # cancels
qc_pass.barrier()
qc_pass.measure_all()

print("Original:")
print(qc_pass.draw())
print_metrics("Original", qc_pass)

Original:
q0: |0>[ H ][ H ][ RX][ RX][ RZ][ RZ]─●──●─ ┆ ─M────
q1: |0>───  ───  ───  ───  ───  ───  ─X──X─ ┆ ────M─
c0:  0                                        ═╩════
c1:  0                                        ════╩═
=== Original ===
qubits: 2
classical_bits: 2
operations: 10
depth: 10
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 2, 'rx': 2, 'rz': 2, 'cx': 2, 'measure': 2}



In [15]:
SELF_INVERSE = {"h", "x", "y", "z", "cx", "cz", "swap"}

def same_location(op1, op2):
    return (
        op1.name == op2.name
        and op1.targets == op2.targets
        and op1.controls == op2.controls
        and op1.condition == op2.condition
    )


def cancel_adjacent_gates(qc: Circuit) -> Circuit:
    stack = []

    for op in qc.operations:
        if op.name == "barrier":
            continue

        if stack:
            prev = stack[-1]

            if (
                op.name in SELF_INVERSE
                and prev.name in SELF_INVERSE
                and same_location(prev, op)
            ):
                stack.pop()
                continue

            if (
                op.name in {"rx", "ry", "rz"}
                and prev.name == op.name
                and prev.targets == op.targets
                and prev.controls == op.controls
                and prev.params
                and op.params
                and abs(prev.params[0] + op.params[0]) < 1e-12
            ):
                stack.pop()
                continue

        stack.append(op)

    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_cancelled")
    for op in stack:
        append_operation(out, op)
    return out


def fuse_adjacent_rotations(qc: Circuit) -> Circuit:
    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_fused")
    i = 0
    ops = qc.operations

    while i < len(ops):
        op = ops[i]

        if (
            op.name in {"rx", "ry", "rz"}
            and len(op.targets) == 1
            and not op.controls
            and op.params
        ):
            axis = op.name
            q = op.targets[0]
            theta = op.params[0]
            j = i + 1

            while j < len(ops):
                nxt = ops[j]
                if (
                    nxt.name == axis
                    and nxt.targets == (q,)
                    and not nxt.controls
                    and nxt.params
                ):
                    theta += nxt.params[0]
                    j += 1
                else:
                    break

            if abs(theta) > 1e-12:
                getattr(out, axis)(theta, q)

            i = j
            continue

        append_operation(out, op)
        i += 1

    return out


def toy_pass_pipeline(qc: Circuit, level=1) -> Circuit:
    out = qc.copy()

    if level >= 1:
        out = cancel_adjacent_gates(out)

    if level >= 2:
        out = fuse_adjacent_rotations(out)

    return out


for level in range(3):
    opt = toy_pass_pipeline(qc_pass, level=level)
    print_metrics(f"Pipeline level {level}", opt)
    print(opt.draw())

=== Pipeline level 0 ===
qubits: 2
classical_bits: 2
operations: 10
depth: 10
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 2, 'rx': 2, 'rz': 2, 'cx': 2, 'measure': 2}

q0: |0>[ H ][ H ][ RX][ RX][ RZ][ RZ]─●──●─ ┆ ─M────
q1: |0>───  ───  ───  ───  ───  ───  ─X──X─ ┆ ────M─
c0:  0                                        ═╩════
c1:  0                                        ════╩═
=== Pipeline level 1 ===
qubits: 2
classical_bits: 2
operations: 4
depth: 3
two_qubit_gates: 0
multi_qubit_gates: 0
gate_counts: {'rz': 2, 'measure': 2}

q0: |0>[ RZ][ RZ]─M────
q1: |0>───  ───  ────M─
c0:  0           ═╩════
c1:  0           ════╩═
=== Pipeline level 2 ===
qubits: 2
classical_bits: 2
operations: 4
depth: 3
two_qubit_gates: 0
multi_qubit_gates: 0
gate_counts: {'rz': 2, 'measure': 2}

q0: |0>[ RZ][ RZ]─M────
q1: |0>───  ───  ────M─
c0:  0           ═╩════
c1:  0           ════╩═


## Subsection 7.3.3 — Mapping and routing on hardware topologies

In [16]:
# Five-qubit circuit with non-adjacent interactions on a line topology.

qc_route = Circuit(5, 5)
qc_route.h(0)
qc_route.cx(0, 4)
qc_route.cx(1, 3)
qc_route.barrier()
qc_route.measure_all()

line_edges_5 = [(0, 1), (1, 2), (2, 3), (3, 4)]

routed = route_cx_to_coupling(qc_route, line_edges_5)

print("Logical circuit:")
print(qc_route.draw())
print_metrics("Logical", qc_route)

print("Routed circuit:")
print(routed.draw())
print_metrics("Routed", routed)

Logical circuit:
q0: |0>[ H ]─●──── ┆ ─M─────────────
q1: |0>───  ────●─ ┆ ────M──────────
q2: |0>───  ────── ┆ ───────M───────
q3: |0>───  ────X─ ┆ ──────────M────
q4: |0>───  ─X──── ┆ ─────────────M─
c0:  0               ═╩═════════════
c1:  0               ════╩══════════
c2:  0               ═══════╩═══════
c3:  0               ══════════╩════
c4:  0               ═════════════╩═
=== Logical ===
qubits: 5
classical_bits: 5
operations: 8
depth: 4
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 1, 'cx': 2, 'measure': 5}

Routed circuit:
q0: |0>[ H ]─x─────────────────x────────── ┆ ─M─────────────
q1: |0>───  ─x──x───────────x──x──x─────x─ ┆ ────M──────────
q2: |0>───  ────x──x─────x──x─────x──●──x─ ┆ ───────M───────
q3: |0>───  ───────x──●──x───────────X──── ┆ ──────────M────
q4: |0>───  ──────────X─────────────────── ┆ ─────────────M─
c0:  0                                       ═╩═════════════
c1:  0                                       ════╩══════════
c2:  0           

## Subsection 7.4.5 — Manual optimization examples

In [17]:
# Manual cancellation of redundant gates.

qc_orig = Circuit(2, 2)
qc_orig.h(0)
qc_orig.cx(0, 1)
qc_orig.cx(0, 1)
qc_orig.z(1)
qc_orig.h(0)
qc_orig.h(0)
qc_orig.barrier()
qc_orig.measure_all()

qc_opt = Circuit(2, 2)
qc_opt.h(0)
qc_opt.z(1)
qc_opt.measure_all()

print("Original:")
print(qc_orig.draw())
print_metrics("Original", qc_orig)

print("Optimized:")
print(qc_opt.draw())
print_metrics("Optimized", qc_opt)

U_orig = circuit_unitary(circuit_without_measurements(qc_orig))
U_opt = circuit_unitary(circuit_without_measurements(qc_opt))

print("Equivalent up to global phase:", equal_up_to_global_phase(U_orig, U_opt))

Original:
q0: |0>[ H ]─●──●────  [ H ][ H ] ┆ ─M────
q1: |0>───  ─X──X─[ Z ]───  ───   ┆ ────M─
c0:  0                              ═╩════
c1:  0                              ════╩═
=== Original ===
qubits: 2
classical_bits: 2
operations: 8
depth: 7
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 3, 'cx': 2, 'z': 1, 'measure': 2}

Optimized:
q0: |0>[ H ]───  ─M────
q1: |0>───  [ Z ]────M─
c0:  0           ═╩════
c1:  0           ════╩═
=== Optimized ===
qubits: 2
classical_bits: 2
operations: 4
depth: 2
two_qubit_gates: 0
multi_qubit_gates: 0
gate_counts: {'h': 1, 'z': 1, 'measure': 2}

Equivalent up to global phase: True


In [18]:
# Rotation fusion.

theta = 0.3
phi = 0.5

qc_rot = Circuit(1)
qc_rot.rz(theta, 0)
qc_rot.rz(phi, 0)
qc_rot.rx(np.pi / 2, 0)
qc_rot.rz(-phi, 0)

qc_fused = Circuit(1)
qc_fused.rz(theta + phi, 0)
qc_fused.rx(np.pi / 2, 0)
qc_fused.rz(-phi, 0)

print("Original:")
print(qc_rot.draw())
print_metrics("Original", qc_rot)

print("Fused:")
print(qc_fused.draw())
print_metrics("Fused", qc_fused)

print(
    "Equivalent up to global phase:",
    equal_up_to_global_phase(
        circuit_unitary(qc_rot),
        circuit_unitary(qc_fused),
    ),
)

Original:
q0: |0>[ RZ][ RZ][ RX][ RZ]
=== Original ===
qubits: 1
classical_bits: 0
operations: 4
depth: 4
two_qubit_gates: 0
multi_qubit_gates: 0
gate_counts: {'rz': 3, 'rx': 1}

Fused:
q0: |0>[ RZ][ RX][ RZ]
=== Fused ===
qubits: 1
classical_bits: 0
operations: 3
depth: 3
two_qubit_gates: 0
multi_qubit_gates: 0
gate_counts: {'rz': 2, 'rx': 1}

Equivalent up to global phase: True


## Subsection 7.5.1 — Topology and coupling constraints

In [19]:
def print_coupling_map(num_qubits, edges):
    print("Qubits:", list(range(num_qubits)))
    print("Coupling edges:", edges)

print_coupling_map(5, line_edges_5)

qc_topology = Circuit(5, 5)
qc_topology.h(0)
qc_topology.cx(0, 4)
qc_topology.cx(1, 3)
qc_topology.barrier()
qc_topology.measure_all()

qc_topology_routed = route_cx_to_coupling(qc_topology, line_edges_5)

print("\nLogical circuit:")
print(qc_topology.draw())

print("\nAfter routing to line topology:")
print(qc_topology_routed.draw())

print_metrics("Logical", qc_topology)
print_metrics("Routed", qc_topology_routed)

Qubits: [0, 1, 2, 3, 4]
Coupling edges: [(0, 1), (1, 2), (2, 3), (3, 4)]

Logical circuit:
q0: |0>[ H ]─●──── ┆ ─M─────────────
q1: |0>───  ────●─ ┆ ────M──────────
q2: |0>───  ────── ┆ ───────M───────
q3: |0>───  ────X─ ┆ ──────────M────
q4: |0>───  ─X──── ┆ ─────────────M─
c0:  0               ═╩═════════════
c1:  0               ════╩══════════
c2:  0               ═══════╩═══════
c3:  0               ══════════╩════
c4:  0               ═════════════╩═

After routing to line topology:
q0: |0>[ H ]─x─────────────────x────────── ┆ ─M─────────────
q1: |0>───  ─x──x───────────x──x──x─────x─ ┆ ────M──────────
q2: |0>───  ────x──x─────x──x─────x──●──x─ ┆ ───────M───────
q3: |0>───  ───────x──●──x───────────X──── ┆ ──────────M────
q4: |0>───  ──────────X─────────────────── ┆ ─────────────M─
c0:  0                                       ═╩═════════════
c1:  0                                       ════╩══════════
c2:  0                                       ═══════╩═══════
c3:  0            

## Subsection 7.5.2 — Native gate sets and basis decomposition

In [20]:
def decompose_to_rx_rz_cx(qc: Circuit) -> Circuit:
    # Rewrite selected logical gates into the basis {rx, rz, cx, measure}.
    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_basis")

    for op in qc.operations:
        if op.name in {"barrier", "measure", "reset"}:
            append_operation(out, op)

        elif op.name == "h":
            q = op.targets[0]
            # H up to global phase.
            out.rz(np.pi/2, q)
            out.rx(np.pi/2, q)
            out.rz(np.pi/2, q)

        elif op.name == "x":
            out.rx(np.pi, op.targets[0])

        elif op.name == "z":
            out.rz(np.pi, op.targets[0])

        elif op.name == "s":
            out.rz(np.pi/2, op.targets[0])

        elif op.name == "t":
            out.rz(np.pi/4, op.targets[0])

        elif op.name == "ry":
            q = op.targets[0]
            theta = op.params[0]
            # RY(theta) = RZ(pi/2) RX(theta) RZ(-pi/2) up to a global phase.
            out.rz(np.pi/2, q)
            out.rx(theta, q)
            out.rz(-np.pi/2, q)

        elif op.name == "cz":
            c = op.controls[0]
            t = op.targets[0]
            out.rz(np.pi/2, t).rx(np.pi/2, t).rz(np.pi/2, t)
            out.cx(c, t)
            out.rz(np.pi/2, t).rx(np.pi/2, t).rz(np.pi/2, t)

        elif op.name in {"rx", "rz", "cx"}:
            append_operation(out, op)

        else:
            append_operation(out, op)

    return out


logical_basis = Circuit(2, 2)
logical_basis.h(0)
logical_basis.ry(0.5, 1)
logical_basis.cz(0, 1)
logical_basis.measure_all()

basis_circuit = decompose_to_rx_rz_cx(logical_basis)

print("Logical circuit:")
print(logical_basis.draw())
print_metrics("Logical", logical_basis)

print("Basis-decomposed circuit:")
print(basis_circuit.draw())
print_metrics("Basis-decomposed", basis_circuit)

Logical circuit:
q0: |0>[ H ]───  ─●─  ─M────
q1: |0>───  [ RY][ CZ]────M─
c0:  0                ═╩════
c1:  0                ════╩═
=== Logical ===
qubits: 2
classical_bits: 2
operations: 5
depth: 3
two_qubit_gates: 1
multi_qubit_gates: 0
gate_counts: {'h': 1, 'ry': 1, 'cz': 1, 'measure': 2}

Basis-decomposed circuit:
q0: |0>[ RZ][ RX][ RZ]───  ───  ───  ───  ───  ───  ─●────  ───  ───  ─M────
q1: |0>───  ───  ───  [ RZ][ RX][ RZ][ RZ][ RX][ RZ]─X─[ RZ][ RX][ RZ]────M─
c0:  0                                                                ═╩════
c1:  0                                                                ════╩═
=== Basis-decomposed ===
qubits: 2
classical_bits: 2
operations: 15
depth: 11
two_qubit_gates: 1
multi_qubit_gates: 0
gate_counts: {'rz': 8, 'rx': 4, 'cx': 1, 'measure': 2}



## Subsection 7.5.3 — Calibration and error-aware adjustments

In [21]:
# A small fake calibration table. Values are illustrative.

fake_calibration = {
    "t1_us": {0: 80, 1: 65, 2: 110, 3: 70, 4: 95},
    "t2_us": {0: 60, 1: 55, 2: 90, 3: 50, 4: 85},
    "single_qubit_error": {0: 0.0008, 1: 0.0011, 2: 0.0005, 3: 0.0014, 4: 0.0007},
    "cx_error": {
        (0, 1): 0.012,
        (1, 2): 0.018,
        (2, 3): 0.009,
        (3, 4): 0.015,
    },
}

for q in range(5):
    print(
        f"Qubit {q}: "
        f"T1={fake_calibration['t1_us'][q]} us, "
        f"T2={fake_calibration['t2_us'][q]} us, "
        f"1q error={fake_calibration['single_qubit_error'][q]:.2e}"
    )

print("\nTwo-qubit error rates:")
for edge, err in fake_calibration["cx_error"].items():
    print(edge, ":", err)


def mapping_score(mapping, calibration):
    # Lower is better. Penalize single-qubit error on used qubits.
    return sum(calibration["single_qubit_error"][q] for q in mapping)


candidate_mappings = [
    [0, 1, 2],
    [1, 2, 3],
    [2, 3, 4],
]

scores = [(m, mapping_score(m, fake_calibration)) for m in candidate_mappings]
scores = sorted(scores, key=lambda x: x[1])

print("\nCandidate logical-to-physical mappings:")
for m, s in scores:
    print(m, "score=", s)

print("\nChosen mapping:", scores[0][0])

Qubit 0: T1=80 us, T2=60 us, 1q error=8.00e-04
Qubit 1: T1=65 us, T2=55 us, 1q error=1.10e-03
Qubit 2: T1=110 us, T2=90 us, 1q error=5.00e-04
Qubit 3: T1=70 us, T2=50 us, 1q error=1.40e-03
Qubit 4: T1=95 us, T2=85 us, 1q error=7.00e-04

Two-qubit error rates:
(0, 1) : 0.012
(1, 2) : 0.018
(2, 3) : 0.009
(3, 4) : 0.015

Candidate logical-to-physical mappings:
[0, 1, 2] score= 0.0024000000000000002
[2, 3, 4] score= 0.0026
[1, 2, 3] score= 0.003

Chosen mapping: [0, 1, 2]


## Subsection 7.5.4 — Balancing fidelity and circuit depth

In [22]:
def estimated_fidelity_from_logical_ops(qc, oneq_error=0.001, twoq_error=0.015, multiq_error=0.05):
    fidelity = 1.0

    for op in qc.operations:
        if op.name in {"barrier", "measure", "reset"}:
            continue

        arity = len(op.targets) + len(op.controls)

        if arity == 1:
            fidelity *= (1 - oneq_error)
        elif arity == 2:
            fidelity *= (1 - twoq_error)
        else:
            fidelity *= (1 - multiq_error)

    return fidelity


qc_depth = Circuit(4, 4)
for q in range(4):
    qc_depth.h(q)
qc_depth.cx(0, 3)
qc_depth.cx(1, 2)
qc_depth.barrier()
qc_depth.measure_all()

qc_depth_routed = route_cx_to_coupling(qc_depth, line_edges_5[:3])

print_metrics("Logical", qc_depth)
print_metrics("Routed", qc_depth_routed)

print(
    "Estimated logical fidelity:",
    estimated_fidelity_from_logical_ops(qc_depth)
)

print(
    "Estimated routed fidelity:",
    estimated_fidelity_from_logical_ops(qc_depth_routed)
)

=== Logical ===
qubits: 4
classical_bits: 4
operations: 10
depth: 4
two_qubit_gates: 2
multi_qubit_gates: 0
gate_counts: {'h': 4, 'cx': 2, 'measure': 4}

=== Routed ===
qubits: 4
classical_bits: 4
operations: 14
depth: 9
two_qubit_gates: 6
multi_qubit_gates: 0
gate_counts: {'h': 4, 'swap': 4, 'cx': 2, 'measure': 4}

Estimated logical fidelity: 0.9663499174700702
Estimated routed fidelity: 0.9096604980080292


## Subsection 7.6.3 — Performance profiling with qcirclab tools

In [23]:
qc_profile = Circuit(5, 5)

for q in range(4):
    qc_profile.h(q)

for q in range(4):
    qc_profile.cx(q, q + 1)

qc_profile.barrier()
qc_profile.measure_all()

start = time.perf_counter()
logical_metrics = circuit_metrics(qc_profile)
elapsed_metrics = time.perf_counter() - start

start = time.perf_counter()
compiled_profile = toy_pass_pipeline(qc_profile, level=2)
compiled_profile = route_cx_to_coupling(compiled_profile, line_edges_5)
elapsed_compile = time.perf_counter() - start

print("=== Logical circuit ===")
print(logical_metrics)

print("\n=== Compiled/routed circuit ===")
print(circuit_metrics(compiled_profile))

print(f"\nMetric extraction time: {elapsed_metrics:.6f} s")
print(f"Toy compilation time: {elapsed_compile:.6f} s")

print("\nCompiled circuit:")
print(compiled_profile.draw())

=== Logical circuit ===
{'qubits': 5, 'classical_bits': 5, 'operations': 13, 'depth': 7, 'two_qubit_gates': 4, 'multi_qubit_gates': 0, 'gate_counts': {'h': 4, 'cx': 4, 'measure': 5}}

=== Compiled/routed circuit ===
{'qubits': 5, 'classical_bits': 5, 'operations': 13, 'depth': 6, 'two_qubit_gates': 4, 'multi_qubit_gates': 0, 'gate_counts': {'h': 4, 'cx': 4, 'measure': 5}}

Metric extraction time: 0.000148 s
Toy compilation time: 0.000307 s

Compiled circuit:
q0: |0>[ H ]───  ───  ───  ─●───────────M─────────────
q1: |0>───  [ H ]───  ───  ─X──●───────────M──────────
q2: |0>───  ───  [ H ]───  ────X──●───────────M───────
q3: |0>───  ───  ───  [ H ]───────X──●───────────M────
q4: |0>───  ───  ───  ───  ──────────X──────────────M─
c0:  0                                 ═╩═════════════
c1:  0                                 ════╩══════════
c2:  0                                 ═══════╩═══════
c3:  0                                 ══════════╩════
c4:  0                                 ═══